# Recomendación de títulos de Netflix: Evaluación

Encontrar películas y series similares por descripción y metadatos.

Diagnóstico descriptivo del catálogo. La coincidencia de categorías es circular como evidencia de calidad porque esas categorías son entradas.

Consulta la [metodología](../docs/metodologia.md). Los reportes corresponden a la última ejecución de cada etapa.


In [1]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("netflix_recommender")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto, notebooks o su carpeta contenedora.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/raw", "data/intermediate", "data/processed", "data/output", "models", "reports"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
from netflix_recommender.alerts import configurar_notebook, mostrar_alertas
configurar_notebook()


In [2]:
from netflix_recommender.flow import run_stage
metricas, comentario = run_stage(PROJECT_ROOT, "evaluacion")
display(Markdown(comentario))
from netflix_recommender.presentation import display_evaluation
display_evaluation(PROJECT_ROOT, metricas)


# Recomendación de títulos de Netflix: evaluacion

Se revisaron **30 consultas** con top 10. La cobertura del catálogo fue **3.41%**. Esta cifra depende de las consultas muestreadas. Sin interacciones ni juicios independientes de relevancia. La coincidencia de categoría no es accuracy ni satisfacción. En Netflix las categorías también son entradas: su coincidencia no constituye una validación independiente.

Si el catálogo tiene menos candidatos que el top solicitado, se devuelven los disponibles; se excluye la consulta y los IDs ya se deduplicaron durante preparación.


,baseline,modelo
consultas,30.000000,30.000000
top_n_solicitado,10.000000,10.000000
cobertura_catalogo,0.001135,0.034064
categorias_distintas_promedio,10.000000,6.700000
coincidencia_categoria_descriptiva,0.310000,0.790000


baseline: consulta s5668


,show_id,title,listed_in,description,similitud
0,s10,The Starling,"Comedies, Dramas",A woman adjusting to life after a loss contend...,None
1,s100,On the Verge,"TV Comedies, TV Dramas","Four women — a chef, a single mom, an heiress ...",None
2,s1000,Stowaway,"Dramas, International Movies, Thrillers",A three-person crew on a mission to Mars faces...,None
3,s1003,Tell Me When,"Comedies, International Movies",Workaholic Will puts his humdrum life in LA on...,None
4,s1004,Zero,"International TV Shows, TV Comedies, TV Dramas",A shy teen with the power to turn invisible mu...,None
5,s1005,Izzy's Koala World,Kids' TV,Young koala caretaker Izzy Bee and her family ...,None
6,s1013,Luis Miguel - The Series,"International TV Shows, Spanish-Language TV Sh...",This series dramatizes the life story of Mexic...,None
7,s102,Untold: Breaking Point,"Documentaries, Sports Movies",Under pressure to continue a winning tradition...,None
8,s1024,Ajeeb Daastaans,"Dramas, Independent Movies, International Movies",Four shorts explore the surprising ways in whi...,None
9,s1025,Arlo the Alligator Boy,"Children & Family Movies, Comedies, Music & Mu...","To find the father he never knew, optimistic A...",None


modelo: consulta s5668


,show_id,title,listed_in,description,similitud
0,s3641,My First First Love,"International TV Shows, Korean TV Shows, Roman...","Due to various personal reasons, a group of Yu...",0.324667
1,s3462,Cheese in the Trap,"International TV Shows, Korean TV Shows, Roman...","In this adaptation of a popular webtoon, a poo...",0.307488
2,s6414,Can We Get Married?,"International TV Shows, Korean TV Shows, Roman...","On the cusp of marriage or divorce, four coupl...",0.306508
3,s5666,Color of Woman,"International TV Shows, Korean TV Shows, Roman...",Two women at a cosmetics company compete for l...,0.302691
4,s1504,Lovestruck in the City,"International TV Shows, Romantic TV Shows, TV ...",Heart stolen by a free-spirited woman after a ...,0.298327
5,s4995,Bad Guys: Vile City,"Crime TV Shows, International TV Shows, Korean...",Ordered to take down a villainous business lea...,0.294062
6,s3475,The Liar and His Lover,"International TV Shows, Korean TV Shows, Roman...",A talented singer falls in love at first sight...,0.287316
7,s2350,It's Okay to Not Be Okay,"International TV Shows, Korean TV Shows, Roman...",An extraordinary road to emotional healing ope...,0.285904
8,s2607,Extracurricular,"Crime TV Shows, International TV Shows, Korean...",A model high school student who's steeped in a...,0.282824
9,s2943,My Holo Love,"International TV Shows, Korean TV Shows, Roman...",Unexpected love finds a lonely woman when she ...,0.282130


## Alertas

Avisos agrupados con descripción y acción sugerida. Los errores no se ocultan; no registrar warnings no garantiza que el análisis sea válido.

In [ ]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/05_evaluacion.json")
from netflix_recommender.informe import generar_informe
generar_informe(PROJECT_ROOT)
